In [1]:
import os
os.chdir('/srv/ext_home/ssalazar/projects/courses/courses/PALOP_2026/Aula_3 Muttações')

# 🧬 Gerador de Sequências com Mutações Reais

Este notebook busca mutações reais de genes humanos no NCBI e cria um CSV.

In [2]:
from Bio import Entrez, SeqIO
from Bio.Seq import Seq
import csv

# Configurar email NCBI
Entrez.email = 'seu_email@exemplo.com'

In [3]:
# Genes por grupo com mutações reais
GENES_MUTACOES = {
    'Grupo 1 - Autossómica Recessiva': {
        'genes': ['CFTR', 'HBB', 'HEXA', 'SMN1'],
        'mutacoes': [
            ('CFTR', 100, 'A', 'G', 'sinonima', 'Fibrose Quística'),
            ('HBB', 20, 'A', 'T', 'nao_sinonima', 'Anemia Falciforme'),
            ('HEXA', 200, 'C', 'G', 'sinonima', 'Doença Tay-Sachs'),
            ('SMN1', 1, 'nao_sinonima', 'Atrofia Muscular', '', ''),
        ]
    },
    'Grupo 2 - Autossómica Dominante': {
        'genes': ['HTT', 'FBN1', 'DMPK', 'NF1'],
        'mutacoes': [
            ('HTT', 68, 'C', 'A', 'nao_sinonima', 'Huntington'),
            ('FBN1', 150, 'G', 'A', 'sinonima', 'Marfan'),
            ('DMPK', 95, 'C', 'T', 'sinonima', 'Distrofia Miotónica'),
            ('NF1', 1, 'nao_sinonima', 'Neurofibromatose', '', ''),
        ]
    },
    'Grupo 3 - Ligada ao X': {
        'genes': ['DMD', 'FMR1', 'F8', 'F9'],
        'mutacoes': [
            ('DMD', 100, 'T', 'G', 'nao_sinonima', 'Duchenne'),
            ('FMR1', 150, 'G', 'A', 'sinonima', 'X Frágil'),
            ('F8', 200, 'A', 'C', 'sinonima', 'Hemofilia A'),
            ('F9', 1, 'nao_sinonima', 'Hemofilia B', '', ''),
        ]
    },
    'Grupo 4 - Cancros Hereditários': {
        'genes': ['BRCA1', 'BRCA2', 'MLH1', 'MSH2'],
        'mutacoes': [
            ('BRCA1', 100, 'T', 'G', 'nao_sinonima', 'BRCA1 mutation'),
            ('BRCA2', 150, 'G', 'A', 'sinonima', 'BRCA2 mutation'),
            ('MLH1', 200, 'C', 'T', 'sinonima', 'Lynch Syndrome'),
            ('MSH2', 1, 'nao_sinonima', 'Lynch Syndrome', '', ''),
        ]
    },
    'Grupo 5 - Metabólica': {
        'genes': ['GBA', 'PAH', 'GALT', 'ABCD1'],
        'mutacoes': [
            ('GBA', 100, 'A', 'G', 'nao_sinonima', 'Gaucher'),
            ('PAH', 150, 'C', 'T', 'sinonima', 'Fenilcetonúria'),
            ('GALT', 200, 'G', 'A', 'sinonima', 'Galactosemia'),
            ('ABCD1', 1, 'nao_sinonima', 'Adrenoleucodistrofia', '', ''),
        ]
    },
    'Grupo 6 - Neurológica': {
        'genes': ['PMP22', 'MPZ', 'TSC1', 'TSC2'],
        'mutacoes': [
            ('PMP22', 100, 'G', 'T', 'nao_sinonima', 'Charcot-Marie-Tooth'),
            ('MPZ', 150, 'A', 'G', 'sinonima', 'Demyelinating neuropathy'),
            ('TSC1', 200, 'T', 'A', 'sinonima', 'Tuberous Sclerosis'),
            ('TSC2', 1, 'nao_sinonima', 'Tuberous Sclerosis', '', ''),
        ]
    }
}

print('Genes carregados com sucesso!')

Genes carregados com sucesso!


In [4]:
def buscar_gene_ncbi(nome_gene, max_tentativas=3):
    '''Busca a sequência de um gene no NCBI.'''
    for tentativa in range(max_tentativas):
        try:
            termo = f'{nome_gene}[Gene Name] AND Homo sapiens[Organism] AND mRNA[Filter]'
            print(f'   Buscando {nome_gene}...', end='', flush=True)
            
            handle = Entrez.esearch(db='nucleotide', term=termo, retmax=1)
            record = Entrez.read(handle)
            handle.close()

            if not record['IdList']:
                print(' nao encontrado')
                continue

            gene_id = record['IdList'][0]
            
            # Aumentar para 10000 bp para ter mais espaço
            handle = Entrez.efetch(
                db='nucleotide', id=gene_id, 
                rettype='fasta', retmode='text',
                seq_start=1, seq_stop=10000
            )
            seq_record = SeqIO.read(handle, 'fasta')
            handle.close()
            
            print(f' OK - {len(seq_record.seq)} bp')
            return str(seq_record.seq), gene_id
            
        except Exception as e:
            print(f' erro')
            continue
    
    return None, None

In [5]:
def aplicar_mutacao(sequencia, posicao, ref, alt):
    '''Aplica uma mutacao numa sequencia.'''
    seq_list = list(sequencia.upper())
    
    # Verificar se a posição está dentro do alcance
    if posicao > len(seq_list) or posicao < 1:
        return None
    
    if seq_list[posicao - 1] != ref:
        return None
    
    seq_list[posicao - 1] = alt
    return ''.join(seq_list)


def is_sinonima(seq_ref, seq_mutada, posicao, ref, alt):
    '''Verifica se uma mutacao e sinonima.'''
    try:
        codon_start = (posicao - 1) // 3 * 3
        codon_end = codon_start + 3
        
        if codon_end > min(len(seq_ref), len(seq_mutada)):
            return None
        
        codon_ref = seq_ref[codon_start:codon_end]
        codon_mut = seq_mutada[codon_start:codon_end]
        
        aa_ref = Seq(codon_ref).translate()
        aa_mut = Seq(codon_mut).translate()
        
        if str(aa_ref) == '*' or str(aa_mut) == '*':
            return False
        
        return str(aa_ref) == str(aa_mut)
    except:
        return None

In [6]:
print('='*80)
print('GERADOR DE SEQUÊNCIAS COM MUTAÇÕES REAIS')
print('='*80)
print(f'Email NCBI: {Entrez.email}\n')

resultados = []

for grupo, info in GENES_MUTACOES.items():
    print(f'\n{grupo}')
    print('-'*80)
    
    genes_lista = info['genes']
    mutacoes_lista = info['mutacoes']
    
    for i, nome_gene in enumerate(genes_lista):
        seq_ref, ncbi_id = buscar_gene_ncbi(nome_gene)
        
        if seq_ref is None:
            print(f'   {nome_gene}: erro')
            continue
        
        gene, posicao, ref, alt, tipo_esperado, doenca = mutacoes_lista[i]
        
        if tipo_esperado == 'nao_sinonima' and doenca == '':
            # Sem mutação
            seq_mutada = seq_ref
            status = 'Referencia'
            print(f'   {nome_gene}: SEM MUTACAO')
        else:
            seq_mutada = aplicar_mutacao(seq_ref, posicao, ref, alt)
            
            if seq_mutada is None:
                print(f'   {nome_gene}: erro na mutacao (posição inválida)')
                seq_mutada = seq_ref
                status = 'Referencia'
            elif tipo_esperado == 'nao_sinonima':
                status = 'Mutado - Nao Sinonima'
                print(f'   {nome_gene}: {posicao}{ref}>{alt} ({doenca})')
            else:
                status = 'Mutado - Sinonima'
                print(f'   {nome_gene}: {posicao}{ref}>{alt} (silent)')
        
        resultados.append({
            'Gene': nome_gene,
            'Grupo': grupo,
            'NCBI_ID': ncbi_id if ncbi_id else '',
            'Comprimento': len(seq_mutada) if seq_mutada else 0,
            'Status': status,
            'Doenca': doenca,
            'Sequencia': seq_mutada if seq_mutada else ''
        })

GERADOR DE SEQUÊNCIAS COM MUTAÇÕES REAIS
Email NCBI: seu_email@exemplo.com


Grupo 1 - Autossómica Recessiva
--------------------------------------------------------------------------------
   Buscando CFTR... OK - 6070 bp
   CFTR: erro na mutacao (posição inválida)
   Buscando HBB... OK - 628 bp
   HBB: erro na mutacao (posição inválida)
   Buscando HEXA... OK - 4785 bp
   HEXA: 200C>G (silent)
   Buscando SMN1... OK - 1571 bp
   SMN1: erro na mutacao (posição inválida)

Grupo 2 - Autossómica Dominante
--------------------------------------------------------------------------------
   Buscando HTT... OK - 10000 bp
   HTT: erro na mutacao (posição inválida)
   Buscando FBN1... OK - 2534 bp
   FBN1: erro na mutacao (posição inválida)
   Buscando DMPK... OK - 2617 bp
   DMPK: 95C>T (silent)
   Buscando NF1... OK - 10000 bp
   NF1: erro na mutacao (posição inválida)

Grupo 3 - Ligada ao X
--------------------------------------------------------------------------------
   Buscando DMD... O

In [7]:
# Salvar em CSV
print('\n'+'='*80)
print(f'RESUMO: {len(resultados)} genes processados')
print('='*80)

nome_ficheiro = 'mutacoes_reais_genes.csv'
with open(nome_ficheiro, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=['Gene', 'Grupo', 'NCBI_ID', 'Comprimento', 'Status', 'Doenca', 'Sequencia'])
    writer.writeheader()
    writer.writerows(resultados)

print(f'\nFicheiro {nome_ficheiro} criado!')
print(f'Nao Sinonimas: {sum(1 for r in resultados if "Nao Sinonima" in r["Status"])}')
print(f'Sinonimas: {sum(1 for r in resultados if "Sinonima" in r["Status"])}')
print(f'Referencia: {sum(1 for r in resultados if r["Status"] == "Referencia")}')


RESUMO: 24 genes processados

Ficheiro mutacoes_reais_genes.csv criado!
Nao Sinonimas: 1
Sinonimas: 4
Referencia: 20


## Converter CSV para FASTA

In [8]:
import pandas as pd

# Ler o CSV
df = pd.read_csv('mutacoes_reais_genes.csv')

print("=" * 80)
print("CONVERTER CSV PARA FASTA")
print("=" * 80)

# Crear ficheiro FASTA
nome_ficheiro_fasta = 'mutacoes_reais_genes.fasta'

with open(nome_ficheiro_fasta, 'w', encoding='utf-8') as f:
    for idx, row in df.iterrows():
        gene = row['Gene']
        status = row['Status']
        doenca = row['Doenca']
        sequencia = row['Sequencia']
        
        # Cabeçalho FASTA: >Gene|Status|Doenca
        header = f">{gene}|{status}|{doenca}"
        f.write(header + "\n")
        
        # Sequência (quebrada a cada 80 caracteres)
        for i in range(0, len(str(sequencia)), 80):
            f.write(str(sequencia)[i:i+80] + "\n")

print(f"\n✅ Ficheiro '{nome_ficheiro_fasta}' criado com sucesso!")
print(f"\n📊 Estatísticas:")
print(f"   Total de sequências: {len(df)}")
print(f"   Não Sinónimas: {sum(df['Status'].str.contains('Nao Sinonima'))}")
print(f"   Sinónimas: {sum(df['Status'].str.contains('Sinonima'))}")
print(f"   Referência: {sum(df['Status'] == 'Referencia')}")

# Mostrar preview do ficheiro
print(f"\n📄 Preview do ficheiro FASTA:")
with open(nome_ficheiro_fasta, 'r', encoding='utf-8') as f:
    linhas = f.readlines()[:10]  # primeiras 10 linhas
    for linha in linhas:
        print(linha.rstrip())

CONVERTER CSV PARA FASTA

✅ Ficheiro 'mutacoes_reais_genes.fasta' criado com sucesso!

📊 Estatísticas:
   Total de sequências: 24
   Não Sinónimas: 1
   Sinónimas: 4
   Referência: 20

📄 Preview do ficheiro FASTA:
>CFTR|Referencia|Fibrose Quística
GTAGTAGGTCTTTGGCATTAGGAGCTTGAGCCCAGACGGCCCTAGCAGGGACCCCAGCGCCCGAGAGACCATGCAGAGGT
CGCCTCTGGAAAAGGCCAGCGTTGTCTCCAAACTTTTTTTCAGCTGGACCAGACCAATTTTGAGGAAAGGATACAGACAG
CGCCTGGAATTGTCAGACATATACCAAATCCCTTCTGTTGATTCTGCTGACAATCTATCTGAAAAATTGGAAAGAGAATG
GGATAGAGAGCTGGCTTCAAAGAAAAATCCTAAACTCATTAATGCCCTTCGGCGATGTTTTTTCTGGAGATTTATGTTCT
ATGGAATCTTTTTATATTTAGGGGAAGTCACCAAAGCAGTACAGCCTCTCTTACTGGGAAGAATCATAGCTTCCTATGAC
CCGGATAACAAGGAGGAACGCTCTATCGCGATTTATCTAGGCATAGGCTTATGCCTTCTCTTTATTGTGAGGACACTGCT
CCTACACCCAGCCATTTTTGGCCTTCATCACATTGGAATGCAGATGAGAATAGCTATGTTTAGTTTGATTTATAAGAAGA
CTTTAAAGCTGTCAAGCCGTGTTCTAGATAAAATAAGTATTGGACAACTTGTTAGTCTCCTTTCCAACAACCTGAACAAA
TTTGATGAAGGACTTGCATTGGCACATTTCGTGTGGATCGCTCCTTTGCAAGTGGCACTCCTCATGGGGCTAATCTGGGA
